In [ ]:
import k3d
import numpy as np
import yaml
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box
from nuscenes.utils.geometry_utils import view_points, box_in_image, BoxVisibility, transform_matrix
from pyquaternion import Quaternion
import os
import sys
from tqdm import tqdm

# --- Add project root to sys.path ---
# (Assuming your notebook is in a 'notebooks' folder at the project root)
NOTEBOOK_DIR = os.path.abspath('') # Gets current directory of the notebook
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- Load Config ---
try:
    with open(os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml'), 'r') as f:
        config = yaml.safe_load(f)
except FileNotFoundError:
    print("Error: config/m_detector_config.yaml not found. Make sure your notebook is in the correct relative path or adjust path.")
    # Provide default dataroot if config fails, for notebook standalone execution
    config = {'nuscenes': {'version': 'v1.0-mini', 'dataroot': '/path/to/your/nuscenes/data'}} # CHANGE THIS

# --- Initialize NuScenes ---
nusc = NuScenes(
    version=config['nuscenes']['version'],
    dataroot=config['nuscenes']['dataroot'],
    verbose=True
)

print("\nNuScenes and helper functions loaded.")

In [ ]:
# Choose a scene (e.g., the first scene in nusc.scene)
# scene_name = "scene-0061" # A common mini scene
# my_scene = nusc.get('scene', nusc.field2token('scene', 'name', scene_name)[0])
my_scene = nusc.scene[1] # Or pick any scene, e.g., scene_index = 1
print(f"Selected Scene: {my_scene['name']} (Description: {my_scene['description']})")

# Get the first sample (keyframe) token in this scene
first_sample_token = my_scene['first_sample_token']
sample = nusc.get('sample', first_sample_token)

# Find an object instance that persists for a few frames
# We need its instance token. Let's look at annotations in the first sample.
instance_token = None
min_num_annotations = 5 # Look for an instance with at least this many annotations across the scene

for ann_token in sample['anns']:
    ann_record = nusc.get('sample_annotation', ann_token)
    instance_rec = nusc.get('instance', ann_record['instance_token'])
    
    # Get the category record using category_token
    category_token = instance_rec['category_token']
    category_record = nusc.get('category', category_token) # Fetch the category record
    category_name = category_record['name'] # Get the name from the category record
            
    if instance_rec['nbr_annotations'] >= min_num_annotations:
        # Check if it's a dynamic object category (e.g., car, pedestrian)
        if 'vehicle' in category_name or 'human' in category_name or 'movable_object' in category_name: # Added movable_object
            instance_token = ann_record['instance_token']
            print(f"\nFound a suitable instance: {instance_token}")
            print(f"Category: {category_name}, Number of annotations in scene: {instance_rec['nbr_annotations']}")
            break

if not instance_token:
    print(f"\nCould not find an instance with at least {min_num_annotations} annotations. Try another scene or lower threshold.")
else:
    instance_record = nusc.get('instance', instance_token) # This is correct

In [ ]:
if instance_token:
    instance_annotations = []
    current_ann_token = instance_record['first_annotation_token']
    for _ in range(instance_record['nbr_annotations']):
        ann_record = nusc.get('sample_annotation', current_ann_token)
        instance_annotations.append(ann_record)
        if not ann_record['next']: # Check if 'next' is empty or None
            break
        current_ann_token = ann_record['next']
    
    # Sort them by timestamp (though they should already be in order)
    instance_annotations.sort(key=lambda ann: nusc.get('sample', ann['sample_token'])['timestamp'])
    
    print(f"\nCollected {len(instance_annotations)} annotations for instance {instance_token}.")
    for i, ann in enumerate(instance_annotations):
        sample_rec = nusc.get('sample', ann['sample_token'])
        print(f"  Ann {i}: sample_token={ann['sample_token']}, timestamp={sample_rec['timestamp']/1e6:.2f}s")

In [ ]:
import k3d
import numpy as np
import os
from nuscenes.nuscenes import NuScenes # Assuming nusc is already initialized from Cell 1
from nuscenes.utils.data_classes import Box
from pyquaternion import Quaternion
from nuscenes.utils.geometry_utils import transform_matrix # For creating transformation matrices

# Define BOX_MESH_INDICES (from your provided example)
# These define the 12 triangles that make up the 6 faces of a cuboid/box.
BOX_MESH_INDICES = np.array([
    [0, 1, 2], [0, 2, 3],  # Front face (example, depends on corner ordering)
    [4, 5, 6], [4, 6, 7],  # Back face
    [0, 1, 5], [0, 5, 4],  # Top face
    [3, 2, 6], [3, 6, 7],  # Bottom face
    [1, 2, 6], [1, 6, 5],  # Right face
    [0, 3, 7], [0, 7, 4]   # Left face
], dtype=np.uint32)
# Note: The exact indices for faces depend on the order of corners from box.corners().
# The nuscenes Box.corners() order is:
# 0: front-left-bottom, 1: front-right-bottom, 2: rear-right-bottom, 3: rear-left-bottom
# 4: front-left-top,    5: front-right-top,    6: rear-right-top,    7: rear-left-top
# Let's redefine BOX_MESH_INDICES based on this standard NuScenes corner ordering for proper faces:
BOX_MESH_INDICES = np.array([
    [0, 1, 2], [0, 2, 3],  # Bottom face
    [4, 5, 6], [4, 6, 7],  # Top face
    [0, 1, 5], [0, 5, 4],  # Front face (0-1-5-4)
    [2, 3, 7], [2, 7, 6],  # Back face (2-3-7-6)
    [1, 2, 6], [1, 6, 5],  # Right face (1-2-6-5)
    [0, 3, 7], [0, 7, 4]   # Left face (0-3-7-4)
], dtype=np.uint32)


if instance_token and instance_annotations: # Ensure these are defined from previous cells
    first_ann = instance_annotations[0]
    sample_record = nusc.get('sample', first_ann['sample_token'])
    
    # --- Get LiDAR Point Cloud (already in sensor frame) ---
    lidar_top_data_token = sample_record['data']['LIDAR_TOP']
    lidar_data_record = nusc.get('sample_data', lidar_top_data_token)
    pcl_path = os.path.join(nusc.dataroot, lidar_data_record['filename'])
    points_sensor_frame = np.fromfile(pcl_path, dtype=np.float32).reshape(-1, 5)[:, :3] # Keep only x,y,z

    # --- Get Transformation Matrices ---
    # Ego vehicle pose in global frame at the time of the LiDAR sweep
    ego_pose_record = nusc.get('ego_pose', lidar_data_record['ego_pose_token'])
    ego_to_global_trans = ego_pose_record['translation']
    ego_to_global_rot = Quaternion(ego_pose_record['rotation'])
    
    # LiDAR sensor pose relative to ego vehicle
    calibrated_sensor_record = nusc.get('calibrated_sensor', lidar_data_record['calibrated_sensor_token'])
    sensor_to_ego_trans = calibrated_sensor_record['translation']
    sensor_to_ego_rot = Quaternion(calibrated_sensor_record['rotation'])

    # --- Create NuScenes Box object (initially in GLOBAL frame) ---
    # The annotation coordinates are in the global frame.
    box_global = Box(
        center=first_ann['translation'],
        size=first_ann['size'], 
        orientation=Quaternion(first_ann['rotation'])
    )
    # Set a name and token for consistency if needed later, though not strictly for plotting here
    box_global.name = nusc.get('category', nusc.get('instance', first_ann['instance_token'])['category_token'])['name']
    box_global.token = first_ann['token']


    # --- Transform the Box to LiDAR Sensor Frame ---
    # 1. Global to Ego Vehicle Frame
    box_global.translate(-np.array(ego_to_global_trans)) # Translate by inverse of ego_global translation
    box_global.rotate(ego_to_global_rot.inverse)       # Rotate by inverse of ego_global rotation
    # Now box_global is effectively box_ego (coordinates are relative to ego vehicle)

    # 2. Ego Vehicle to LiDAR Sensor Frame
    box_global.translate(-np.array(sensor_to_ego_trans)) # Translate by inverse of sensor_ego translation
    box_global.rotate(sensor_to_ego_rot.inverse)       # Rotate by inverse of sensor_ego rotation
    # Now box_global is effectively box_lidar (coordinates are relative to LiDAR sensor)
    box_lidar_frame = box_global # Rename for clarity, it's the same object modified in place

    # Get corners of the box, now in LiDAR sensor frame
    corners_lidar_frame = box_lidar_frame.corners().T.astype(np.float32) # 8x3 array of vertices

    # --- k3d Plot Setup (inspired by your working example) ---
    plot = k3d.plot(camera_auto_fit=False, grid_visible=False, menu_visibility=True, axes_helper=0.1) # axes_helper for origin
    # Using menu_visibility=True for now to help debug if needed

    # Add LiDAR points (already in sensor frame)
    # Downsample for performance if necessary
    points_to_plot = points_sensor_frame
    if points_sensor_frame.shape[0] > 70000: 
        print(f"Downsampling point cloud from {points_sensor_frame.shape[0]} to 70000 points for visualization.")
        choice_indices = np.random.choice(points_sensor_frame.shape[0], 70000, replace=False)
        points_to_plot = points_sensor_frame[choice_indices, :]
        
    plot += k3d.points(
        positions=points_to_plot.astype(np.float32),
        point_size=0.05, 
        color=0xaaaaaa, # Light grey
        name='LiDAR Points (Sensor Frame)'
    )
    
    # Add the annotation box as a mesh (now in sensor frame)
    # Get color for the box (example: red, or use NuScenes colormap)
    try:
        category_color_rgb = nusc.colormap[box_lidar_frame.name] # This is (R,G,B) in 0-255
        box_color_k3d = (category_color_rgb[0] << 16) + (category_color_rgb[1] << 8) + category_color_rgb[2]
    except KeyError:
        box_color_k3d = 0xff0000 # Default to red if category name not in colormap

    plot += k3d.mesh(
        vertices=corners_lidar_frame, # 8x3 array of corner coordinates
        indices=BOX_MESH_INDICES,     # 12x3 array defining triangles for the box faces
        color=box_color_k3d,
        opacity=0.5, # Make it slightly transparent
        name=f'Annotation: {box_lidar_frame.name}'
    )
    
    # Set camera (copied from your example, adjust if needed or use camera_auto_fit=True initially)
    # If camera_auto_fit=False, a good initial camera position is crucial.
    # The example camera might be specific to a certain view or data.
    # Let's try camera_auto_fit=True first, as it's more general.
    # plot.camera_auto_fit = True
    # If auto_fit doesn't work, you can uncomment and use a fixed camera like your example:
    plot.camera = [14.323085106370606, -28.314391114259966, 5.941020835869653, 10.157863789244125, -0.1506512943750446, -3.7441543221541664, 0.029062043355090668, 0.274372196368901, 0.9504079626643153] # Example camera

    # Display the plot
    plot.display()
    
    print(f"\nVisualizing annotation for instance {instance_token[:6]} in LiDAR SENSOR frame.")
    print(f"  Box Name: {box_lidar_frame.name}")
    print(f"  Box Center (in LiDAR frame): {box_lidar_frame.center}")
    print(f"  Box WLH: {box_lidar_frame.wlh}")
    print(f"  Number of LiDAR points in sweep: {points_sensor_frame.shape[0]}")

else:
    print("No instance or annotations selected/available to visualize.")

In [ ]:
# Cell 5: Calculate Interpolated Box States (Position, Orientation, Velocity)

import numpy as np
from pyquaternion import Quaternion
from nuscenes.utils.data_classes import Box

# --- Helper function for Box interpolation ---
def interpolate_box(box1: Box, box2: Box, ratio: float) -> Box:
    """
    Interpolates between two Box objects.
    - Linearly interpolates center, size (wlh), and velocity.
    - Spherically interpolates orientation (quaternion).

    Args:
        box1 (Box): The starting box.
        box2 (Box): The ending box.
        ratio (float): The interpolation ratio (0.0 for box1, 1.0 for box2).

    Returns:
        Box: A new Box object representing the interpolated state.
    """
    if not (0.0 <= ratio <= 1.0):
        raise ValueError("Interpolation ratio must be between 0.0 and 1.0.")

    # LERP for center
    interp_center = box1.center * (1 - ratio) + box2.center * ratio
    
    # LERP for size (wlh)
    # Box stores size as wlh internally
    interp_wlh = np.array(box1.wlh) * (1 - ratio) + np.array(box2.wlh) * ratio
    
    # SLERP for orientation
    # Ensure box1.orientation and box2.orientation are pyquaternion.Quaternion objects
    # The Box class constructor already ensures this if 'rotation' was a list/array.
    interp_orientation = Quaternion.slerp(box1.orientation, box2.orientation, ratio)
    
    # LERP for velocity
    # Ensure velocities are numpy arrays
    vel1 = np.array(box1.velocity)
    vel2 = np.array(box2.velocity)
    interp_velocity = vel1 * (1 - ratio) + vel2 * ratio
    
    # Create the new interpolated Box object
    # The Box constructor expects 'size' as [width, length, height]
    interpolated_b = Box(center=interp_center.tolist(),
                         size=interp_wlh.tolist(), 
                         orientation=interp_orientation,
                         velocity=interp_velocity.tolist())
    
    # Optionally, carry over name and token if they are consistent
    # This depends on how you want to treat these attributes for interpolated boxes.
    # For now, let's assume they might not be directly applicable or could be set later.
    # interpolated_b.name = box1.name # Or some other logic
    # interpolated_b.token = None # Interpolated boxes don't have their own annotation tokens

    return interpolated_b

# --- Sanity checks and setup ---
if 'nusc' not in locals():
    print("NuScenes SDK (nusc) not initialized. Please run Cell 1.")
elif 'instance_token' not in locals() or 'instance_annotations' not in locals():
    print("Instance token or annotations not found. Please run Cells 2 and 3.")
elif not instance_annotations:
    print(f"Instance {instance_token} has no annotations.")
elif len(instance_annotations) < 2:
    print(f"Instance {instance_token} has only {len(instance_annotations)} annotation(s). Need at least 2 for interpolation.")
else:
    print(f"Found {len(instance_annotations)} sorted annotations for instance {instance_token[:6]}.")
    print("Attempting to interpolate between the first two annotations.")

    ann_k1 = instance_annotations[0]
    ann_k2 = instance_annotations[1]

    sample_k1_token = ann_k1['sample_token']
    sample_k2_token = ann_k2['sample_token']
    sample_k1_rec = nusc.get('sample', sample_k1_token)
    sample_k2_rec = nusc.get('sample', sample_k2_token)
    ts_k1 = sample_k1_rec['timestamp'] 
    ts_k2 = sample_k2_rec['timestamp']

    if ts_k1 >= ts_k2:
        print(f"Error: Timestamp of first keyframe's sample ({ts_k1}) is not less than the second's ({ts_k2}).")
    else:
        print(f"Keyframe 1 (K1): Ann token {ann_k1['token'][:6]}, Sample token {sample_k1_token[:6]}, Timestamp: {ts_k1 / 1e6:.3f}s")
        print(f"Keyframe 2 (K2): Ann token {ann_k2['token'][:6]}, Sample token {sample_k2_token[:6]}, Timestamp: {ts_k2 / 1e6:.3f}s")
        
        velocity_k1 = nusc.box_velocity(ann_k1['token'])
        box_k1 = Box(center=ann_k1['translation'],
                     size=ann_k1['size'], 
                     orientation=Quaternion(ann_k1['rotation']),
                     velocity=velocity_k1)
        box_k1.token = ann_k1['token']
        category_k1_token = nusc.get('instance', ann_k1['instance_token'])['category_token']
        box_k1.name = nusc.get('category', category_k1_token)['name']

        velocity_k2 = nusc.box_velocity(ann_k2['token'])
        box_k2 = Box(center=ann_k2['translation'],
                     size=ann_k2['size'],
                     orientation=Quaternion(ann_k2['rotation']),
                     velocity=velocity_k2)
        box_k2.token = ann_k2['token']
        category_k2_token = nusc.get('instance', ann_k2['instance_token'])['category_token']
        box_k2.name = nusc.get('category', category_k2_token)['name']

        print(f"\nBox K1 ({box_k1.name}):")
        print(f"  Center (global): {np.round(box_k1.center, 2)}")
        print(f"  Velocity (global m/s): {np.round(box_k1.velocity, 2)}")
        print(f"  Size (W,L,H): {np.round(box_k1.wlh, 2)}")
        print(f"  Orientation (Quaternion w,x,y,z): {np.round(box_k1.orientation.elements, 3)}")

        print(f"Box K2 ({box_k2.name}):")
        print(f"  Center (global): {np.round(box_k2.center, 2)}")
        print(f"  Velocity (global m/s): {np.round(box_k2.velocity, 2)}")
        print(f"  Size (W,L,H): {np.round(box_k2.wlh, 2)}")
        print(f"  Orientation (Quaternion w,x,y,z): {np.round(box_k2.orientation.elements, 3)}")

        num_interp_steps = 3
        all_boxes_in_sequence = [box_k1]
        all_timestamps_in_sequence = [ts_k1]

        print(f"\nCalculating {num_interp_steps} interpolated states between K1 and K2:")
        for i in range(1, num_interp_steps + 1):
            time_diff_ratio = i / (num_interp_steps + 1.0)
            actual_interpolated_timestamp = ts_k1 * (1 - time_diff_ratio) + ts_k2 * time_diff_ratio
            all_timestamps_in_sequence.append(actual_interpolated_timestamp)
            
            # Use our new helper function
            interpolated_box = interpolate_box(box_k1, box_k2, time_diff_ratio)
            # Assign the name from box1 for consistency in this sequence
            interpolated_box.name = box_k1.name 
            all_boxes_in_sequence.append(interpolated_box)

            print(f"\n  Interpolated Step {i}:")
            print(f"    Time Ratio (from K1 to K2): {time_diff_ratio:.3f}")
            print(f"    Effective Timestamp: {actual_interpolated_timestamp / 1e6:.3f}s")
            print(f"    Center (global): {np.round(interpolated_box.center, 2)}")
            print(f"    Velocity (global m/s): {np.round(interpolated_box.velocity, 2)}")
            print(f"    Size (W,L,H): {np.round(interpolated_box.wlh, 2)}")
            print(f"    Orientation (Quaternion w,x,y,z): {np.round(interpolated_box.orientation.elements, 3)}")
        
        all_boxes_in_sequence.append(box_k2)
        all_timestamps_in_sequence.append(ts_k2)
        
        print(f"\nTotal boxes in sequence (K1 + interpolated + K2): {len(all_boxes_in_sequence)}")

In [ ]:
# Cell 6: Plot Interpolated Sequence (Bird's-Eye View using Matplotlib)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon # For drawing the box as a filled shape
from IPython.display import HTML # For displaying animation in Jupyter

# --- Ensure previous cell's data is available ---
if 'all_boxes_in_sequence' not in locals() or 'all_timestamps_in_sequence' not in locals():
    print("Interpolated box sequence not found. Please run Cell 5 first.")
    # You might want to raise an error or handle this more robustly
elif not all_boxes_in_sequence:
    print("Box sequence is empty. Cannot plot.")
else:
    print(f"Found {len(all_boxes_in_sequence)} boxes in the sequence to animate.")

    # --- Determine plot limits to fit all boxes in the sequence ---
    # Initialize with broad limits, then narrow down
    min_x_coord, max_x_coord = float('inf'), float('-inf')
    min_y_coord, max_y_coord = float('inf'), float('-inf')

    for box_obj in all_boxes_in_sequence:
        # box.bottom_corners() returns 3x4 array (x,y,z rows; 4 corner columns)
        # We only need the x (row 0) and y (row 1) coordinates
        corners_xy = box_obj.bottom_corners()[:2, :] 
        
        current_min_x = np.min(corners_xy[0, :])
        current_max_x = np.max(corners_xy[0, :])
        current_min_y = np.min(corners_xy[1, :])
        current_max_y = np.max(corners_xy[1, :])
        
        if current_min_x < min_x_coord: min_x_coord = current_min_x
        if current_max_x > max_x_coord: max_x_coord = current_max_x
        if current_min_y < min_y_coord: min_y_coord = current_min_y
        if current_max_y > max_y_coord: max_y_coord = current_max_y

    # Add some padding to the limits for better visualization
    padding = 5.0  # meters; adjust as needed
    plot_xlim = (min_x_coord - padding, max_x_coord + padding)
    plot_ylim = (min_y_coord - padding, max_y_coord + padding)
    
    # --- Setup the plot ---
    fig, ax = plt.subplots(figsize=(10, 10)) # Adjust figsize for a good aspect ratio
    
    # Initialize a Polygon artist for the box. This will be updated in each frame.
    # The Polygon needs a list/array of (x,y) vertices.
    # box_obj.bottom_corners()[:2, :].T gives a 4x2 array of (x,y) for the 4 bottom corners.
    initial_box_corners_bev = all_boxes_in_sequence[0].bottom_corners()[:2, :].T
    box_patch = Polygon(initial_box_corners_bev, closed=True, edgecolor='royalblue', facecolor='skyblue', alpha=0.7, linewidth=1.5)
    ax.add_patch(box_patch)

    # Initialize a line artist for the orientation (front of the box)
    # We'll draw a line from the box center towards its front.
    initial_box = all_boxes_in_sequence[0]
    # Local x-axis (front) vector scaled by half length, transformed to global
    front_vec_local = np.array([initial_box.wlh[1] / 2.0, 0, 0]) # [length/2, 0, 0]
    front_vec_global = initial_box.orientation.rotation_matrix @ front_vec_local
    line_start_x, line_start_y = initial_box.center[0], initial_box.center[1]
    line_end_x, line_end_y = initial_box.center[0] + front_vec_global[0], initial_box.center[1] + front_vec_global[1]
    orientation_line, = ax.plot([line_start_x, line_end_x], [line_start_y, line_end_y], color='red', linewidth=2)

    # Text for timestamp and frame number
    time_text_artist = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10, 
                               bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.5))

    # --- Animation update function ---
    def update(frame_num):
        current_box = all_boxes_in_sequence[frame_num]
        current_timestamp_us = all_timestamps_in_sequence[frame_num] # in microseconds

        # Update polygon vertices for the current box
        bev_corners = current_box.bottom_corners()[:2, :].T
        box_patch.set_xy(bev_corners)
        
        # Update orientation line
        front_vec_local_update = np.array([current_box.wlh[1] / 2.0, 0, 0])
        front_vec_global_update = current_box.orientation.rotation_matrix @ front_vec_local_update
        new_line_start_x, new_line_start_y = current_box.center[0], current_box.center[1]
        new_line_end_x, new_line_end_y = current_box.center[0] + front_vec_global_update[0], current_box.center[1] + front_vec_global_update[1]
        orientation_line.set_data([new_line_start_x, new_line_end_x], [new_line_start_y, new_line_end_y])
        
        # Update timestamp text
        # Display relative time from the start of the sequence for clarity
        relative_time_s = (current_timestamp_us - all_timestamps_in_sequence[0]) / 1e6
        time_text_artist.set_text(f'Frame: {frame_num}\nTime: {relative_time_s:+.3f}s')
        
        # Set plot attributes (these are static but good to ensure they are set)
        # If not using blit=True or if ax.cla() was used, these would be essential each frame.
        # With blit=True and updating artists, they are mostly for initial setup.
        ax.set_xlim(plot_xlim)
        ax.set_ylim(plot_ylim)
        ax.set_xlabel("Global X (meters)")
        ax.set_ylabel("Global Y (meters)")
        ax.set_title(f"BEV of Interpolated Box: '{current_box.name}' (Instance: {instance_token[:6]})")
        ax.set_aspect('equal', adjustable='box') # Crucial for BEV
        ax.grid(True, linestyle='--', alpha=0.7)
        
        return box_patch, orientation_line, time_text_artist # Artists to be redrawn

    # --- Create and display animation ---
    # Interval: delay between frames in ms. 
    # Our interpolated steps are effectively at 20 Hz (0.05s or 50ms apart).
    # For smoother viewing, we can increase the interval.
    # e.g., if num_interp_steps = 9, we have 11 total frames spanning 0.5s.
    # Each step is 0.5s / 10 intervals = 0.05s = 50ms.
    animation_interval_ms = 100 # milliseconds (e.g., 10 frames per second display rate)
    
    # `blit=True` means only re-draw the parts that have changed.
    anim = FuncAnimation(fig, update, frames=len(all_boxes_in_sequence), 
                         interval=animation_interval_ms, blit=True)

    # Close the static plot that Matplotlib might create by default before showing the animation
    plt.close(fig) 
    
    # Display the animation in Jupyter
    # Using to_jshtml() is often more robust in different Jupyter environments.
    display(HTML(anim.to_jshtml())) 
    # Alternatively, for a more video-like player (might need ffmpeg for some browsers/notebooks):
    # display(HTML(anim.to_html5_video()))

    # --- Optional: Save the animation ---
    # To save as GIF (requires imagemagick to be installed and discoverable by Matplotlib)
    # gif_path = "interpolated_sequence_bev.gif"
    # anim.save(gif_path, writer='imagemagick', fps=1000/animation_interval_ms)
    # print(f"Animation saved to {gif_path}")

    # To save as MP4 (requires ffmpeg to be installed and discoverable by Matplotlib)
    # mp4_path = "interpolated_sequence_bev.mp4"
    # anim.save(mp4_path, writer='ffmpeg', fps=1000/animation_interval_ms)
    # print(f"Animation saved to {mp4_path}")

In [ ]:
# Cell 7: Plot Interpolated Sequence with LiDAR Context (BEV using Matplotlib)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon
from IPython.display import HTML
import os

from nuscenes.utils.data_classes import LidarPointCloud
from nuscenes.utils.geometry_utils import transform_matrix
from pyquaternion import Quaternion

# --- Ensure previous cell's data is available ---
if 'nusc' not in locals() or 'all_boxes_in_sequence' not in locals() or \
   'all_timestamps_in_sequence' not in locals() or \
   'ann_k1' not in locals() or 'ann_k2' not in locals() or \
   'instance_token' not in locals():
    print("Required data not found. Please run Cells 1, 3, and 5 first.")
elif not all_boxes_in_sequence:
    print("Box sequence is empty. Cannot plot.")
else:
    print(f"Found {len(all_boxes_in_sequence)} box states and timestamps to animate.")

    # --- 1. Identify Relevant LiDAR Sweeps for the interval ---
    sample_k1_rec = nusc.get('sample', ann_k1['sample_token'])
    sample_k2_rec = nusc.get('sample', ann_k2['sample_token'])
    
    ts_k1_us = sample_k1_rec['timestamp'] # Microseconds
    ts_k2_us = sample_k2_rec['timestamp'] # Microseconds

    relevant_lidar_sds = [] # List to store (timestamp_us, sample_data_token)
    
    # Start with the LIDAR_TOP data from the first keyframe's sample
    current_sd_token = sample_k1_rec['data']['LIDAR_TOP']
    
    # Traverse forward through LIDAR_TOP sample_data records
    # until we pass the timestamp of the second keyframe's sample
    max_lidar_scans_to_collect = 30 # Safety break for the loop
    scans_collected = 0
    
    while current_sd_token and scans_collected < max_lidar_scans_to_collect:
        sd_rec = nusc.get('sample_data', current_sd_token)
        relevant_lidar_sds.append({'timestamp_us': sd_rec['timestamp'], 'token': current_sd_token, 'filename': sd_rec['filename']})
        
        if sd_rec['timestamp'] >= ts_k2_us and sd_rec['next']: # Ensure we have at least one scan at or after ts_k2
             # Add one more scan beyond ts_k2 to ensure coverage for the last interpolated points
            next_sd_rec = nusc.get('sample_data', sd_rec['next'])
            relevant_lidar_sds.append({'timestamp_us': next_sd_rec['timestamp'], 'token': next_sd_rec['token'], 'filename': next_sd_rec['filename']})
            break
        elif not sd_rec['next']: # Reached the end of the scene/channel
            break
            
        current_sd_token = sd_rec['next']
        scans_collected += 1

    if not relevant_lidar_sds:
        print("Error: Could not find any relevant LiDAR sweeps.")
        # Exit or raise error
    else:
        print(f"Collected {len(relevant_lidar_sds)} LiDAR sample_data records for the interval.")

    # Sort by timestamp just in case traversal order wasn't perfect (it should be)
    relevant_lidar_sds.sort(key=lambda x: x['timestamp_us'])
    
    # --- Helper function to get closest LiDAR sweep ---
    def get_closest_lidar_sd(target_timestamp_us):
        closest_sd = min(relevant_lidar_sds, key=lambda sd: abs(sd['timestamp_us'] - target_timestamp_us))
        return closest_sd

    # --- Determine plot limits (can be expanded by point clouds) ---
    min_x_coord, max_x_coord = float('inf'), float('-inf')
    min_y_coord, max_y_coord = float('inf'), float('-inf')

    for box_obj in all_boxes_in_sequence:
        corners_xy = box_obj.bottom_corners()[:2, :]
        min_x_coord = min(min_x_coord, np.min(corners_xy[0, :]))
        max_x_coord = max(max_x_coord, np.max(corners_xy[0, :]))
        min_y_coord = min(min_y_coord, np.min(corners_xy[1, :]))
        max_y_coord = max(max_y_coord, np.max(corners_xy[1, :]))
    
    padding = 15.0  # Increased padding for point cloud
    plot_xlim = (min_x_coord - padding, max_x_coord + padding)
    plot_ylim = (min_y_coord - padding, max_y_coord + padding)
    
    # --- Setup the plot ---
    fig, ax = plt.subplots(figsize=(8, 8))
    
    initial_box_corners_bev = all_boxes_in_sequence[0].bottom_corners()[:2, :].T
    box_patch = Polygon(initial_box_corners_bev, closed=True, edgecolor='royalblue', facecolor='skyblue', alpha=0.8, linewidth=1.5, zorder=10)
    ax.add_patch(box_patch)

    initial_box = all_boxes_in_sequence[0]
    front_vec_local = np.array([initial_box.wlh[1] / 2.0, 0, 0])
    front_vec_global = initial_box.orientation.rotation_matrix @ front_vec_local
    orientation_line, = ax.plot([initial_box.center[0], initial_box.center[0] + front_vec_global[0]],
                                [initial_box.center[1], initial_box.center[1] + front_vec_global[1]],
                                color='red', linewidth=2, zorder=11)

    # Initialize scatter plot for LiDAR points (will be updated)
    # zorder places it below the box
    lidar_scatter = ax.scatter([], [], s=1.5, c='dimgray', alpha=0.6, zorder=1) 

    time_text_artist = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10,
                               bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.7), zorder=12)

    # --- LiDAR Point Cloud Loading and Transformation Cache (simple version) ---
    # To avoid reloading/recomputing for the exact same lidar scan if it's used multiple times
    lidar_cache = {} 

    # --- Animation update function ---
    def update(frame_num):
        current_box = all_boxes_in_sequence[frame_num]
        target_timestamp_us = all_timestamps_in_sequence[frame_num]

        # Update box polygon
        bev_corners = current_box.bottom_corners()[:2, :].T
        box_patch.set_xy(bev_corners)
        
        # Update orientation line
        front_vec_local_update = np.array([current_box.wlh[1] / 2.0, 0, 0])
        front_vec_global_update = current_box.orientation.rotation_matrix @ front_vec_local_update
        orientation_line.set_data([current_box.center[0], current_box.center[0] + front_vec_global_update[0]],
                                  [current_box.center[1], current_box.center[1] + front_vec_global_update[1]])
        
        # --- Get and plot LiDAR data ---
        closest_lidar_info = get_closest_lidar_sd(target_timestamp_us)
        lidar_sd_token = closest_lidar_info['token']

        if lidar_sd_token in lidar_cache:
            points_global_bev = lidar_cache[lidar_sd_token]
        else:
            lidar_sd_rec = nusc.get('sample_data', lidar_sd_token)
            pcl_path = os.path.join(nusc.dataroot, lidar_sd_rec['filename'])
            
            if not os.path.exists(pcl_path):
                print(f"LiDAR file not found: {pcl_path}")
                points_global_bev = np.array([[],[]]).T # Empty
            else:
                # Load points (sensor frame)
                pc = LidarPointCloud.from_file(pcl_path)
                points_sensor_frame = pc.points[:3, :] # Shape (3, N)
                
                # Get sensor pose relative to ego
                cs_rec = nusc.get('calibrated_sensor', lidar_sd_rec['calibrated_sensor_token'])
                sensor_to_ego_tf = transform_matrix(cs_rec['translation'], Quaternion(cs_rec['rotation']))
                
                # Get ego pose relative to global
                ego_pose_rec = nusc.get('ego_pose', lidar_sd_rec['ego_pose_token'])
                ego_to_global_tf = transform_matrix(ego_pose_rec['translation'], Quaternion(ego_pose_rec['rotation']))
                
                # Transform points: sensor -> ego -> global
                points_sensor_homogeneous = np.vstack((points_sensor_frame, np.ones(points_sensor_frame.shape[1])))
                points_global_homogeneous = ego_to_global_tf @ sensor_to_ego_tf @ points_sensor_homogeneous
                points_global = points_global_homogeneous[:3, :]
                
                # Downsample for plotting performance (e.g., take every Nth point)
                downsample_factor = 20 # Adjust as needed for performance/density
                points_global_bev = points_global[:2, ::downsample_factor].T # Get (x,y) and transpose to (N,2)
            
            lidar_cache[lidar_sd_token] = points_global_bev
            # print(f"Frame {frame_num}: Loaded LiDAR {lidar_sd_token[:6]} (TS: {closest_lidar_info['timestamp_us']/1e6:.3f}s) for target TS {target_timestamp_us/1e6:.3f}s. Points: {points_global_bev.shape[0]}")


        if points_global_bev.shape[0] > 0:
            lidar_scatter.set_offsets(points_global_bev)
        else:
            lidar_scatter.set_offsets(np.array([[],[]]).T) # Clear points if none

        relative_time_s = (target_timestamp_us - all_timestamps_in_sequence[0]) / 1e6
        time_text_artist.set_text(f'Frame: {frame_num}\nTime: {relative_time_s:+.3f}s\nLiDAR TS: {closest_lidar_info["timestamp_us"]/1e6:.3f}s')
        
        ax.set_xlim(plot_xlim)
        ax.set_ylim(plot_ylim)
        ax.set_xlabel("Global X (meters)")
        ax.set_ylabel("Global Y (meters)")
        ax.set_title(f"BEV of '{current_box.name}' (Instance: {instance_token[:6]}) with LiDAR")
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, linestyle='--', alpha=0.6)
        
        return box_patch, orientation_line, time_text_artist, lidar_scatter

    # --- Create and display animation ---
    animation_interval_ms = 150 # May need to increase if LiDAR loading is slow
    
    anim = FuncAnimation(fig, update, frames=len(all_boxes_in_sequence), 
                         interval=animation_interval_ms, blit=True)

    plt.close(fig)
    display(HTML(anim.to_jshtml()))
    
    # Optional: Save
    # gif_path = "interpolated_sequence_bev_with_lidar.gif"
    # anim.save(gif_path, writer='imagemagick', fps=max(1, int(1000/animation_interval_ms))) # fps should be > 0
    # print(f"Animation saved to {gif_path}")

In [ ]:
# --- Ensure previous cells' data and functions are available ---
if 'nusc' not in locals():
    print("NuScenes SDK (nusc) not initialized. Please run Cell 1.")
    raise RuntimeError("nusc not initialized") 
elif 'my_scene' not in locals(): 
    print("Variable 'my_scene' (the scene record) not found. Please run Cell 2.")
    raise RuntimeError("my_scene not initialized") 
elif 'instance_token' not in locals(): 
    print("Variable 'instance_token' not found. Please run Cell 2.")
    raise RuntimeError("instance_token not initialized") 
elif 'interpolate_box' not in locals(): 
    print("The 'interpolate_box' helper function is not defined. Please ensure Cell 5 was run successfully.")
    raise RuntimeError("interpolate_box not defined") 
else:
    print("Starting full scene animation setup...")

    # --- 1. Scene and Timeline Setup ---
    scene_rec = my_scene 
    scene_token = scene_rec['token'] 

    print(f"Working with scene: {scene_rec['name']} (Token: {scene_token[:6]}, Description: {scene_rec['description']})")

    current_sample_token = scene_rec['first_sample_token']
    scene_samples = [] 
    while current_sample_token:
        sample_rec = nusc.get('sample', current_sample_token)
        scene_samples.append(sample_rec)
        current_sample_token = sample_rec['next']
    
    scene_samples.sort(key=lambda s: s['timestamp']) 
    
    if not scene_samples:
        print(f"Error: No samples found for scene {scene_token}.")
        raise RuntimeError(f"No samples for scene {scene_token}") 
    else:
        scene_start_time_us = scene_samples[0]['timestamp']
        scene_end_time_us = scene_samples[-1]['timestamp']
        scene_duration_s = (scene_end_time_us - scene_start_time_us) / 1e6
        print(f"Scene duration: {scene_duration_s:.2f}s (from {scene_start_time_us} to {scene_end_time_us} us)")

        TARGET_HZ = 20.0
        dt_us = int(1e6 / TARGET_HZ) 
        target_timestamps_us = np.arange(scene_start_time_us, scene_end_time_us + dt_us, dt_us)
        num_animation_frames = len(target_timestamps_us)
        print(f"Generated {num_animation_frames} target timestamps for {TARGET_HZ}Hz animation.")

        # --- 2. Data Pre-computation ---
        all_animation_frame_data = [] 
        lidar_points_global_cache = {} 
        
        all_lidar_sds_in_scene = []
        for sample in scene_samples:
            sd_token = sample['data'].get('LIDAR_TOP') 
            if sd_token:
                sd_rec_lidar = nusc.get('sample_data', sd_token) 
                all_lidar_sds_in_scene.append(sd_rec_lidar)
        all_lidar_sds_in_scene.sort(key=lambda sd: sd['timestamp']) 
        lidar_sds_timestamps_us_array = np.array([sd['timestamp'] for sd in all_lidar_sds_in_scene])

        print(f"\nStarting pre-computation of {num_animation_frames} animation frames...")
        precomp_start_time = time.time()

        for frame_idx, current_target_ts_us in enumerate(target_timestamps_us):
            if frame_idx % max(1, (num_animation_frames // 20)) == 0 or frame_idx == num_animation_frames - 1:
                elapsed_time = time.time() - precomp_start_time
                print(f"  Processing frame {frame_idx + 1}/{num_animation_frames} (Target TS: {current_target_ts_us / 1e6:.3f}s). Elapsed: {elapsed_time:.1f}s")

            sample_timestamps_us_array = np.array([s['timestamp'] for s in scene_samples])
            idx_next_sample = np.searchsorted(sample_timestamps_us_array, current_target_ts_us, side='right')
            idx_prev_sample = idx_next_sample - 1

            if idx_next_sample == 0: 
                sample_prev = scene_samples[0]
                sample_next = scene_samples[0]
            elif idx_prev_sample >= len(scene_samples) - 1: 
                sample_prev = scene_samples[-1]
                sample_next = scene_samples[-1]
            else:
                sample_prev = scene_samples[idx_prev_sample]
                sample_next = scene_samples[idx_next_sample]

            ts_sample_prev_us = sample_prev['timestamp']
            ts_sample_next_us = sample_next['timestamp']

            if ts_sample_next_us > ts_sample_prev_us:
                interp_ratio_kf = (current_target_ts_us - ts_sample_prev_us) / (ts_sample_next_us - ts_sample_prev_us)
            else: 
                interp_ratio_kf = 0.0 if current_target_ts_us <= ts_sample_prev_us else 1.0
            interp_ratio_kf = np.clip(interp_ratio_kf, 0.0, 1.0)

            sd_prev_lidar_token = sample_prev['data'].get('LIDAR_TOP')
            sd_next_lidar_token = sample_next['data'].get('LIDAR_TOP')

            if not sd_prev_lidar_token or not sd_next_lidar_token: 
                print(f"Warning: Missing LIDAR_TOP data for bracketing samples for TS {current_target_ts_us}. Using last known good ego pose.")
                if not all_animation_frame_data: 
                    first_sample_sd_token = scene_samples[0]['data'].get('LIDAR_TOP')
                    if not first_sample_sd_token: raise RuntimeError("Cannot get ego pose, even for first sample.")
                    first_sample_sd_rec = nusc.get('sample_data', first_sample_sd_token)
                    first_ego_pose_rec = nusc.get('ego_pose', first_sample_sd_rec['ego_pose_token'])
                    current_ego_pose_global_tf = transform_matrix(first_ego_pose_rec['translation'], Quaternion(first_ego_pose_rec['rotation']))
                else: 
                    current_ego_pose_global_tf = all_animation_frame_data[-1]['ego_pose_global_tf']
            else:
                ego_pose_prev_rec = nusc.get('ego_pose', nusc.get('sample_data', sd_prev_lidar_token)['ego_pose_token'])
                ego_pose_next_rec = nusc.get('ego_pose', nusc.get('sample_data', sd_next_lidar_token)['ego_pose_token'])
                ego_transl_prev = np.array(ego_pose_prev_rec['translation'])
                ego_transl_interp = ego_transl_prev * (1 - interp_ratio_kf) + np.array(ego_pose_next_rec['translation']) * interp_ratio_kf
                ego_rot_prev = Quaternion(ego_pose_prev_rec['rotation'])
                ego_rot_interp = Quaternion.slerp(ego_rot_prev, Quaternion(ego_pose_next_rec['rotation']), interp_ratio_kf)
                current_ego_pose_global_tf = transform_matrix(ego_transl_interp, ego_rot_interp)

            points_global_for_frame = np.empty((3,0)) 
            closest_lidar_sd_timestamp_us = -1

            if len(lidar_sds_timestamps_us_array) > 0:
                closest_lidar_idx = np.argmin(np.abs(lidar_sds_timestamps_us_array - current_target_ts_us))
                closest_lidar_sd_rec_for_frame = all_lidar_sds_in_scene[closest_lidar_idx] 
                closest_lidar_sd_token = closest_lidar_sd_rec_for_frame['token']
                closest_lidar_sd_timestamp_us = closest_lidar_sd_rec_for_frame['timestamp']

                if closest_lidar_sd_token in lidar_points_global_cache:
                    points_global_for_frame = lidar_points_global_cache[closest_lidar_sd_token]
                else:
                    pcl_path = os.path.join(nusc.dataroot, closest_lidar_sd_rec_for_frame['filename'])
                    if os.path.exists(pcl_path):
                        pc = LidarPointCloud.from_file(pcl_path)
                        points_sensor_frame = pc.points[:3, :] 
                        cs_rec = nusc.get('calibrated_sensor', closest_lidar_sd_rec_for_frame['calibrated_sensor_token'])
                        sensor_to_ego_tf = transform_matrix(cs_rec['translation'], Quaternion(cs_rec['rotation']))
                        ego_pose_of_scan_rec = nusc.get('ego_pose', closest_lidar_sd_rec_for_frame['ego_pose_token'])
                        ego_of_scan_to_global_tf = transform_matrix(ego_pose_of_scan_rec['translation'], Quaternion(ego_pose_of_scan_rec['rotation']))
                        points_sensor_homogeneous = np.vstack((points_sensor_frame, np.ones(points_sensor_frame.shape[1])))
                        points_global_homogeneous = ego_of_scan_to_global_tf @ sensor_to_ego_tf @ points_sensor_homogeneous
                        points_global_for_frame = points_global_homogeneous[:3, :]
                        lidar_points_global_cache[closest_lidar_sd_token] = points_global_for_frame
            
            interpolated_boxes_global_for_frame = []
            anns_prev_tokens = sample_prev['anns']
            anns_next_tokens = sample_next['anns']
            anns_prev_recs_by_instance = {nusc.get('sample_annotation', tk)['instance_token']: nusc.get('sample_annotation', tk) for tk in anns_prev_tokens}
            anns_next_recs_by_instance = {nusc.get('sample_annotation', tk)['instance_token']: nusc.get('sample_annotation', tk) for tk in anns_next_tokens}

            for inst_tk, ann_p_rec in anns_prev_recs_by_instance.items():
                if inst_tk in anns_next_recs_by_instance: 
                    ann_n_rec = anns_next_recs_by_instance[inst_tk]
                    vel_p = nusc.box_velocity(ann_p_rec['token'])
                    if np.any(np.isnan(vel_p)): vel_p = np.array([0.0, 0.0, 0.0])
                    vel_n = nusc.box_velocity(ann_n_rec['token'])
                    if np.any(np.isnan(vel_n)): vel_n = np.array([0.0, 0.0, 0.0])
                    box_p = Box(center=ann_p_rec['translation'], size=ann_p_rec['size'], 
                                orientation=Quaternion(ann_p_rec['rotation']), velocity=vel_p)
                    box_n = Box(center=ann_n_rec['translation'], size=ann_n_rec['size'], 
                                orientation=Quaternion(ann_n_rec['rotation']), velocity=vel_n)
                    interp_box_global = interpolate_box(box_p, box_n, interp_ratio_kf)
                    
                    # ******** CORRECTED LINE HERE ********
                    interp_box_global.name = ann_p_rec['category_name'] 
                    # ******** END CORRECTION ********
                                        
                    interp_box_global.token = inst_tk 
                    interpolated_boxes_global_for_frame.append(interp_box_global)
            
            all_animation_frame_data.append({
                'target_timestamp_us': current_target_ts_us,
                'ego_pose_global_tf': current_ego_pose_global_tf, 
                'lidar_points_global': points_global_for_frame,   
                'lidar_sd_timestamp_us': closest_lidar_sd_timestamp_us, 
                'boxes_global': interpolated_boxes_global_for_frame 
            })
        
        precomp_end_time = time.time()
        print(f"\nPre-computation for {num_animation_frames} frames finished in {precomp_end_time - precomp_start_time:.2f} seconds.")

        # --- 3. Matplotlib Animation ---
        # (The Matplotlib animation setup and update function remains the same as the previous correct version)
        print("Setting up Matplotlib animation...")
        fig, ax = plt.subplots(figsize=(10, 10)) 

        EGO_PLOT_RANGE_M = 50.0 
        ax.set_xlim(-EGO_PLOT_RANGE_M, EGO_PLOT_RANGE_M)
        ax.set_ylim(-EGO_PLOT_RANGE_M, EGO_PLOT_RANGE_M)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel("X relative to Ego (meters)")
        ax.set_ylabel("Y relative to Ego (meters)")
        ax.grid(True, linestyle='--', alpha=0.5)

        lidar_scatter_ego_frame = ax.scatter([], [], s=1.0, c='dimgray', alpha=0.6, zorder=1)
        ego_vertices = np.array([[-1.5, -0.8], [-1.5, 0.8], [2.0, 0.0]]) * 1.0 
        ego_marker_patch = MatplotlibPolygon(ego_vertices, closed=True, facecolor='darkorange', edgecolor='black', zorder=20)
        ax.add_patch(ego_marker_patch) 

        time_text_artist = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=9,
                                   bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.7), zorder=22)

        MAX_BOXES_TO_DISPLAY_SIMULTANEOUSLY = 100 
        box_patches_ego_frame = []
        for i in range(MAX_BOXES_TO_DISPLAY_SIMULTANEOUSLY):
            p = MatplotlibPolygon([[0,0]], closed=True, edgecolor='blue', facecolor='cyan', 
                                  alpha=0.5, linewidth=1.0, visible=False, zorder=10) 
            ax.add_patch(p)
            box_patches_ego_frame.append(p)
            
        category_colors = {
            'vehicle.car': ('green', 'lightgreen'), 'human.pedestrian.adult': ('red', 'lightcoral'),
            'vehicle.bicycle': ('blue', 'lightblue'), 'vehicle.motorcycle': ('deepskyblue', 'paleturquoise'),
            'vehicle.truck': ('black', 'darkgray'), 'vehicle.bus.rigid': ('purple', 'plum'),
            'movable_object.trafficcone': ('gold', 'khaki'), 
        }
        default_box_color_edge, default_box_color_face = ('gray', 'silver')
        
        DOWNSAMPLE_LIDAR_FOR_ANIMATION = 30 

        def update_animation_frame(frame_idx_anim):
            current_frame_data = all_animation_frame_data[frame_idx_anim]
            ego_pose_global_tf_this_frame = current_frame_data['ego_pose_global_tf']
            global_to_current_ego_tf = np.linalg.inv(ego_pose_global_tf_this_frame)
            
            points_global = current_frame_data['lidar_points_global']
            if points_global.shape[1] > 0: 
                points_global_downsampled = points_global[:, ::DOWNSAMPLE_LIDAR_FOR_ANIMATION]
                points_global_hom = np.vstack((points_global_downsampled, np.ones(points_global_downsampled.shape[1])))
                points_ego_hom = global_to_current_ego_tf @ points_global_hom
                points_ego_bev = points_ego_hom[:2, :].T 
                lidar_scatter_ego_frame.set_offsets(points_ego_bev)
            else:
                lidar_scatter_ego_frame.set_offsets(np.array([[],[]]).T) 

            visible_boxes_this_frame = 0
            for i, box_global_obj in enumerate(current_frame_data['boxes_global']):
                if i < MAX_BOXES_TO_DISPLAY_SIMULTANEOUSLY:
                    corners_global = box_global_obj.corners()
                    corners_global_hom = np.vstack((corners_global, np.ones(corners_global.shape[1])))
                    corners_ego_hom = global_to_current_ego_tf @ corners_global_hom
                    bev_corners_ego_frame = corners_ego_hom[:2, :4].T 
                    box_patches_ego_frame[i].set_xy(bev_corners_ego_frame)
                    edge_color, face_color = category_colors.get(box_global_obj.name, 
                                                                 (default_box_color_edge, default_box_color_face))
                    box_patches_ego_frame[i].set_edgecolor(edge_color)
                    box_patches_ego_frame[i].set_facecolor(face_color)
                    box_patches_ego_frame[i].set_visible(True)
                    visible_boxes_this_frame += 1
                else:
                    break 
            
            for k in range(visible_boxes_this_frame, MAX_BOXES_TO_DISPLAY_SIMULTANEOUSLY):
                box_patches_ego_frame[k].set_visible(False)

            current_scene_time_s = (current_frame_data['target_timestamp_us'] - scene_start_time_us) / 1e6
            actual_lidar_time_s = (current_frame_data['lidar_sd_timestamp_us'] - scene_start_time_us) / 1e6 \
                                  if current_frame_data['lidar_sd_timestamp_us'] > 0 else -1.0
            
            title_str = f"Scene: {scene_rec['name']} (Ego-Centric BEV @ {TARGET_HZ:.0f}Hz)"
            if 'instance_token' in locals() and instance_token: # Check if instance_token is defined
                 title_str += f"\nFocus Instance (context): {instance_token[:6]}"

            ax.set_title(title_str) 
            time_text_artist.set_text(f'Frame: {frame_idx_anim + 1}/{num_animation_frames}\n'
                                      f'Scene Time: {current_scene_time_s:.2f}s / {scene_duration_s:.2f}s\n'
                                      f'LiDAR (rel. TS): {actual_lidar_time_s:.2f}s')
            
            return [lidar_scatter_ego_frame, ego_marker_patch, time_text_artist] + box_patches_ego_frame

        animation_interval_ms = max(20, int(1000 / TARGET_HZ)) 
        anim = FuncAnimation(fig, update_animation_frame, frames=num_animation_frames, 
                             interval=animation_interval_ms, blit=True)

        plt.close(fig) 
        print("\nDisplaying animation in Jupyter Notebook (this may take a moment to render)...")
        display(HTML(anim.to_jshtml()))